# Antibody Generation with AntibodyGPT
This notebook is a companion of chapter 8 of the "Domain Specific LLMs in Action" book, author Guglielmo Iozzia, [Manning Publications](https://www.manning.com/), 2024.  
The code in this notebook is to generate antibody sequences using the [AntibodyGPT](https://huggingface.co/AntibodyGeneration/fine-tuned-progen2-small) model. It requires hardware acceleration.  
More details about the code can be found in the related book's chapter.

Downgrading the HF's Transformers library to ensure compatibility with the AntibodyGPT2's `ProGenForCausalLM` class, as it inherits from `PreTrainedModel`, which, starting from Transformers release 4.50, wouldn't inherit from `GenerationMixin` anymore, in so loosing the availability of the `generate` method.

In [1]:
!pip install transformers==4.49.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/44.0 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 106.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


Clone the official repo.

In [2]:
!git clone https://github.com/joethequant/docker_protein_generator.git
%cd ./docker_protein_generator/

Cloning into 'docker_protein_generator'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 29 (delta 8), reused 25 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 20.23 KiB | 20.23 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/docker_protein_generator


Download one of the pretrained models and the associated tokenizer from the HF's Hub. Please note that the AutoClass to use is the custom ```ProGenForCausalLM``` available in the ```docker_protein_generator``` cloned repo.



In [3]:
from models.progen.modeling_progen import ProGenForCausalLM
import torch
from tokenizers import Tokenizer

model_path = 'AntibodyGeneration/fine-tuned-progen2-small'

model = ProGenForCausalLM.from_pretrained(model_path)
tokenizer = Tokenizer.from_pretrained(model_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/969 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/617M [00:00<?, ?B/s]

ProGenForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
from pathlib import Path

models_path = Path("antibodygen")
model.save_pretrained(models_path)

The save model to disk is 588.6 MB (617 MB in memory after download).

Define a target antigen sequence and the number of antibody sequences you want to generate for it and then start the generation process.

In [5]:
target_sequence = 'MQIPQAPWPVVWAVLQLGWRPGWFLDSPDRPWNPPTFSPALLVVTEGDNATFTCSFSNTSESFVLNWYRMSPSNQTDKLAAFPEDRSQPGQDCRFRVTQLPNGRDFHMSVVRARRNDSGTYLCGAISLAPKAQIKESLRAELRVTERRAEVPTAHPSPSPRPAGQFQTLVVGVVGGLLGSLVLLVWVLAVICSRAARGTIGARRTGQPLKEDPSAVPVFSVDYGELDFQWREKTPEPPVPCVPEQTEYATIVFPSGMGTSSPARRGSADGPRSAQPLRPEDGHCSWPL'
number_of_sequences = 2

Tokenize the prompt sequence and then convert it to PyTorch tensor and move it to the GPU.

In [6]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
tokenized_sequence = tokenizer.encode(target_sequence)
input_tensor = torch.tensor([tokenized_sequence.ids]).to(device)

Move the model to GPU.

In [7]:
model = model.to(device)

Start the sequence generation.

In [ ]:
with torch.no_grad():
    output = model.generate(input_tensor,
                            max_length=1024,
                            pad_token_id=tokenizer.encode('<|pad|>').ids[0],
	                        do_sample=True,
                            top_p=0.9,
                            temperature=0.8,
	                        num_return_sequences=number_of_sequences)


In [9]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 151,148,576
Trainable Parameters: 151,148,576


It is a classic irony in deep learning: **a smaller model does not always mean faster inference.** While `AntibodyGeneration/fine-tuned-progen2-small` has a relatively modest 160M parameters, its slowness compared to other models of its size (like a BERT-Base or a small GPT-2) usually boils down to the **Autoregressive Bottleneck** and the specific **Sampling overhead** required for biological validity.

Here is why that specific model feels heavy:

---

### 1. The Autoregressive "Tax" (Sequential Generation)
Unlike "Encoder-only" models (like ESM-2 or BERT) that process an entire sequence in a single mathematical pass, ProGen2 is a **Decoder-only** model.
* **Token-by-Token:** To generate a 100-residue antibody, the model must run 100 separate forward passes.
* **KV-Caching Overhead:** For each new amino acid, the model must attend to all previous ones. As the sequence grows, the computational cost of the "Attention" mechanism increases linearly, slowing down the later stages of the generation.



### 2. High Sampling Complexity
In your previous question, we looked at parameters like `top_k`, `repetition_penalty`, and `do_sample`.
* **The Logits Processor:** Before the model can pick the next amino acid, it must run all your "Warpers" (repetition penalty, top-k, temperature).
* **Small Vocabulary, Big Sort:** Even though the vocabulary is small (~25–30 tokens), performing complex stochastic sampling and penalty calculations 100+ times per sequence adds significant CPU-bound latency that offsets the GPU-bound speed of the 160M parameters.


Decode the generated sequences and display them.

In [10]:
as_lists = lambda batch: [batch[i, ...].detach().cpu().numpy().tolist() for i in range(batch.shape[0])]
sequences = tokenizer.decode_batch(as_lists(output))
if len(sequences) > 0:
    sequences = [x.replace('2', '') for x in sequences]

In [11]:
sequences

['MQIPQAPWPVVWAVLQLGWRPGWFLDSPDRPWNPPTFSPALLVVTEGDNATFTCSFSNTSESFVLNWYRMSPSNQTDKLAAFPEDRSQPGQDCRFRVTQLPNGRDFHMSVVRARRNDSGTYLCGAISLAPKAQIKESLRAELRVTERRAEVPTAHPSPSPRPAGQFQTLVVGVVGGLLGSLVLLVWVLAVICSRAARGTIGARRTGQPLKEDPSAVPVFSVDYGELDFQWREKTPEPPVPCVPEQTEYATIVFPSGMGTSSPARRGSADGPRSAQPLRPEDGHCSWPLAPARTRFQEVEPKVQPAGRALGPHPALGALEPHHYAGQHHGFCAPKAPKPPAPMGLLQPPRLGGALVLPCLAGYGPSPEFSWSSHNRYLSVKRADINPYLLHTDNVLQFLSVEERDAEQFWGTGSFTCTATQPATKVARAAVRPGRPSYDRHSHHHHHHGGGGSGGGGSGGGGSQVQLVQSGAEVKKPGASVKVSCKASGYTFTGYTMHWVRQAPGQGLEWMGFINSYTGDISSAQKFQGRVTMTRDTSISTAYMELSRLRSDDTAVYYCARGTRMRSVEGGFDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKRVEPKSCGGGGSGGGGSGGGGSEIVLTQSPGTLSLSPGERATLSCRASQSVSSRYLAWYQQKPGQAPRLLIYGASSRAPGIPDRFSGSGSGTDFTLTISRLEPEDFAVYYCQQYGSSPWTFGQGTKVEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLRSPVTKSFNRGEC',
 'MQIPQAPWPVVWAVLQLGWRPGWFLDSPDRPWNPPTFSPALLVVTEGDNATFTCSFSNTSESFVLNWYRMSPSNQT